# Module 3 v2 — 3-class U-Net (background / strawberry / peduncle)

Trains on **ROI crops and 3-class masks** from `multiclass_masks/` (Module 2 v2 output).

Improvements over v1: multi-class head, Cross-Entropy + multi-class Dice, **early stopping patience=5**, **randomised stratified split** by contributor, **stronger augmentations** (flip, rotation, colour jitter).

Outputs: `runs/unet_v2/best_unet_v2.pt`, training curves, prediction grid, **3×3 confusion matrix** on validation pixels.

## Cell 1 — Imports and setup

Loads libraries, sets hyperparameters (`EPOCHS=60`, `BATCH=8`, `LR0=3e-4`, `IMG_SIZE=256`, `SEED=42`), scans `multiclass_masks/` for image–mask pairs, and confirms the device (CUDA/CPU).  
**Prerequisite:** `multiclass_masks/` must be populated by `module2_v2_multiclass_masks.ipynb`.  
**Expected output:** device and pair count.

In [ ]:
%matplotlib inline
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

PROJECT  = r"C:\Users\markm\Documents\RDP\AIDA 2158\Final Project"
IMGS     = os.path.join(PROJECT, "multiclass_masks", "images")
MASKS    = os.path.join(PROJECT, "multiclass_masks", "masks_3class")
RUNS     = os.path.join(PROJECT, "runs", "unet_v2")
os.makedirs(RUNS, exist_ok=True)
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_C    = 3
IMG_SIZE = 256
BATCH    = 8
EPOCHS   = 50
PATIENCE = 5
LR0      = 0.001
SEED     = 42

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

all_imgs  = sorted(glob.glob(os.path.join(IMGS, "*.png")))
all_lbls  = [os.path.join(MASKS, os.path.splitext(os.path.basename(p))[0] + ".png") for p in all_imgs]
all_lbls  = [p if os.path.exists(p) else None for p in all_lbls]
pairs = [(i, m) for i, m in zip(all_imgs, all_lbls) if m is not None]
print(f"Device: {DEVICE} | Found {len(pairs)} image–mask pairs in multiclass_masks")

## Cell 2 — Train / val split (stratified by contributor)

Assigns a contributor label to each image pair and uses stratified `train_test_split` (16% val, seed=42) so all four annotators appear in both splits. Saves the split indices and validation filenames to `runs/unet_v2/val_split_v2.json` (required by Module 4 v2).  
**Expected output:** confirms train/val counts.

In [ ]:
def contributor_of(path: str) -> str:
    b = os.path.basename(path)
    for p in ("hervejunior_", "biberdork_", "aida2154_", "markm_"):
        if b.startswith(p):
            return p.rstrip("_")
    return "other"

y_strat   = [contributor_of(p[0]) for p in pairs]
idx       = list(range(len(pairs)))
try:
    tr_idx, va_idx = train_test_split(idx, test_size=0.16, random_state=SEED, stratify=y_strat)
except ValueError:
    tr_idx, va_idx = train_test_split(idx, test_size=0.16, random_state=SEED)
train_pairs = [pairs[i] for i in tr_idx]
val_pairs   = [pairs[i] for i in va_idx]
import json as _json
os.makedirs(RUNS, exist_ok=True)
_json.dump({"tr_idx": tr_idx, "va_idx": va_idx, "val_names": [os.path.basename(p[0]) for p in val_pairs]},
           open(os.path.join(RUNS, "val_split_v2.json"), "w"), indent=2)
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)} (stratified by contributor) — split saved to runs/unet_v2/val_split_v2.json")

## Cell 3 — Dataset class (Roi3Dataset)

PyTorch Dataset that loads ROI crops and 3-class masks from `multiclass_masks/`. Training mode applies augmentation: horizontal flip (50%), small rotation ±15° (40%), HSV colour jitter (40%). All images resized to 256×256 and normalised with ImageNet stats.

In [ ]:
class Roi3Dataset(Dataset):
    def __init__(self, pair_list, augment: bool = False):
        self.pair_list = pair_list
        self.augment   = augment
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self): return len(self.pair_list)

    def __getitem__(self, i):
        ip, mp = self.pair_list[i]
        img  = np.array(Image.open(ip).convert("RGB"))
        m    = np.array(Image.open(mp).convert("L"))
        m    = np.clip(m.astype(np.int32), 0, 2).astype(np.uint8)
        h, w = m.shape
        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy(); m = np.fliplr(m).copy()
            if random.random() < 0.4:
                ang = random.uniform(-15, 15); M = cv2.getRotationMatrix2D((w/2, h/2), ang, 1.0)
                img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
                m   = cv2.warpAffine(m,   M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            if random.random() < 0.4:
                hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
                hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-8, 8)) % 180
                hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.85, 1.15), 0, 255)
                hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.85, 1.15), 0, 255)
                img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        m_t = torch.from_numpy(m).long().unsqueeze(0).unsqueeze(0).float()
        m_t = F.interpolate(m_t, size=(IMG_SIZE, IMG_SIZE), mode='nearest').long().squeeze(0).squeeze(0)
        p = Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        t  = transforms.ToTensor()(p)
        t  = self.norm(t)
        return t, m_t

print("Roi3Dataset defined — augmented (flip/rotate/HSV) training, resizes to 256×256, returns (tensor, LongTensor mask).")

## Cell 4 — UNet3 architecture and loss functions

Defines `UNet3` (3-class, 4-level encoder-decoder with BatchNorm and skip connections) and two helper functions:
- `soft_dice_multiclass` — multi-class soft Dice loss (combined with CrossEntropy during training).
- `iou_per_class` — computes per-class IoU from logits (used for validation metrics).

In [ ]:
def double_conv(in_c, out_c, bias=True):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, padding=1, bias=bias),
        nn.BatchNorm2d(out_c), nn.ReLU(True),
        nn.Conv2d(out_c, out_c, 3, padding=1, bias=bias),
        nn.BatchNorm2d(out_c), nn.ReLU(True),
    )

class UNet3(nn.Module):
    def __init__(self, in_ch=3, n_classes=3, feats=None):
        super().__init__()
        feats = feats or [64, 128, 256, 512]
        self.encs = nn.ModuleList(); ch = in_ch
        for f in feats:
            self.encs.append(double_conv(ch, f, False)); ch = f
        self.pool = nn.MaxPool2d(2, 2)
        self.bott = double_conv(feats[-1], feats[-1] * 2, False)
        self.ups = nn.ModuleList(); self.decs = nn.ModuleList()
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.decs.append(double_conv(f*2, f, False))
        self.out = nn.Conv2d(feats[0], n_classes, 1)

    def forward(self, x):
        sk = []
        for e in self.encs:
            x = e(x); sk.append(x); x = self.pool(x)
        x = self.bott(x); sk = sk[::-1]
        for i, (up, dec) in enumerate(zip(self.ups, self.decs)):
            x = up(x); s = sk[i]
            if x.shape != s.shape: x = F.interpolate(x, s.shape[2:])
            x = dec(torch.cat([s, x], 1))
        return self.out(x)

def soft_dice_multiclass(logits, target, n_c=3):
    prob = F.softmax(logits, dim=1); loss = 0.0; eps=1e-6
    for c in range(n_c):
        p = prob[:, c]; t = (target == c).float()
        inter = (p * t).sum(); den = p.sum() + t.sum() + eps
        loss += 1.0 - (2*inter + eps) / den
    return loss / n_c

def iou_per_class(logits, target, n_c=3):
    pred = logits.argmax(1); ious = []
    for c in range(n_c):
        p_ = (pred == c); t_ = (target == c)
        inter = (p_ & t_).sum().float(); union = (p_ | t_).sum().float() + 1e-6
        ious.append((inter/union).item())
    return ious, float(np.mean(ious))

print("UNet3 defined (3-class, 4-level encoder-decoder, BatchNorm). Loss helpers: soft_dice_multiclass, iou_per_class.")

## Cell 5 — Training loop

Initialises DataLoaders, model (UNet3), Adam optimiser, and cosine LR scheduler. Trains for up to `EPOCHS` with **early stopping (patience=5)** on validation mIoU. Best weights saved to `runs/unet_v2/best_unet_v2.pt`.  
Prints per-epoch: loss, mIoU, per-class IoU. Stops early if no improvement.

**Weighted loss:** `CrossEntropyLoss` is given inverse-frequency class weights computed from the training masks. Because background pixels outnumber peduncle pixels ~23:1, without weighting the model learns to ignore peduncle. Weights are computed dynamically so they adapt to whatever crop distribution module2_v2 produced (printed at cell start).

In [ ]:
assert len(train_pairs) > 0, 'Run module2_v2_multiclass_masks.ipynb first to populate multiclass_masks/'
train_loader = DataLoader(Roi3Dataset(train_pairs, True),  batch_size=BATCH, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(Roi3Dataset(val_pairs, False), batch_size=BATCH, shuffle=False, num_workers=0)
model  = UNet3().to(DEVICE)
# Compute inverse-frequency class weights from training masks so the loss
# doesn't ignore the rare peduncle class (class 2).
_counts = np.zeros(NUM_C, dtype=np.int64)
for _, mp in train_pairs:
    _m = np.array(Image.open(mp).convert('L'))
    for c in range(NUM_C): _counts[c] += int(np.sum(_m == c))
_total = _counts.sum()
_w = _total / (NUM_C * _counts.astype(np.float64))
_w = _w / _w.min()                         # normalize so smallest weight = 1.0
_w = np.clip(_w, 1.0, 12.0).astype(np.float32)  # cap minority class at 12x max
ce_weight = torch.tensor(_w).to(DEVICE)
ce     = nn.CrossEntropyLoss(weight=ce_weight)
print(f"Class weights  BG={_w[0]:.2f}  Fruit={_w[1]:.2f}  Peduncle={_w[2]:.2f}")
opt    = optim.Adam(model.parameters(), lr=LR0)
sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
best_m = -1.0; best_ep = 0; stall = 0; hist = []

def run_epoch(loader, train_mode=False):
    if train_mode: model.train()
    else: model.eval()
    tot_loss = 0.0; n = 0; sum_iou = 0.0; nb = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train_mode):
            log = model(x)
            L = ce(log, y) + soft_dice_multiclass(log, y, NUM_C)
            if train_mode:
                opt.zero_grad(); L.backward(); opt.step()
        tot_loss += L.item() * x.size(0); n += x.size(0)
        with torch.no_grad():
            iou_l, m_iou = iou_per_class(log, y, NUM_C)
        sum_iou += m_iou; nb += 1
    m_iou = sum_iou / max(nb, 1)
    return tot_loss / max(n,1), m_iou

@torch.no_grad()
def per_class_iou(loader):
    model.eval(); inter_ = [0,0,0]; uni_ = [0,0,0]
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = model(x).argmax(1)
        for c in range(3): inter_[c] += ((p==c)&(y==c)).sum().item(); uni_[c] += ((p==c)|(y==c)).sum().item()
    return [inter_[c] / max(uni_[c], 1) for c in range(3)]

def confusion_full(loader):
    model.eval(); cm = np.zeros((NUM_C, NUM_C), dtype=np.int64)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            p = model(x).argmax(1)
            for t in range(NUM_C):
                for s in range(NUM_C):
                    cm[t, s] += ((y == t) & (p == s)).sum().item()
    return cm

for ep in range(1, EPOCHS+1):
    tr_l, tr_m = run_epoch(train_loader, True)
    va_l, _ = run_epoch(val_loader, False)
    va_pc = per_class_iou(val_loader)
    va_m = float(np.mean(va_pc))
    sched.step()
    hist.append((tr_l, tr_m, va_l, va_m, va_pc))
    print(f"Ep {ep:02d} | tr {tr_l:.4f} tr_mIoU {tr_m:.4f} | val {va_l:.4f} val_mIoU {va_m:.4f}  perCls { [round(x,3) for x in va_pc]}")
    if va_m > best_m:
        best_m = va_m; best_ep = ep; stall = 0
        torch.save(model.state_dict(), os.path.join(RUNS, "best_unet_v2.pt"))
    else:
        stall += 1
    if stall >= PATIENCE and ep > 5:
        print(f"Early stop at {ep} (no mIoU gain {PATIENCE} eps). Best {best_m:.4f} @ ep {best_ep}")
        break
model.load_state_dict(torch.load(os.path.join(RUNS, 'best_unet_v2.pt'), map_location=DEVICE))
cm = confusion_full(val_loader)
np.savetxt(os.path.join(RUNS, "confusion_val.csv"), cm, fmt="%d", delimiter=",")
print("Confusion matrix (rows=GT, cols=pred):\n", cm)
fig, ax = plt.subplots()
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_xticklabels(['bg','fruit','ped']); ax.set_yticklabels(['bg','fruit','ped'])
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
plt.title('Pixel confusion (val) — U-Net v2'); plt.tight_layout(); plt.savefig(os.path.join(RUNS, 'unet_v2_confusion_matrix.png'), dpi=130); plt.show()
print('Saved confusion matrix →', os.path.join(RUNS, 'unet_v2_confusion_matrix.png'))

## Cell 6 — Confusion matrix

Builds a pixel-level 3×3 confusion matrix (background / strawberry / peduncle) on the validation set using the best saved weights. Prints the matrix and saves the figure.  
Saves → `runs/unet_v2/unet_v2_confusion_matrix.png` and `confusion_val.csv`.

In [ ]:
if hist:
    epn = list(range(1, len(hist)+1))
    tr_l = [h[0] for h in hist]; tr_m = [h[1] for h in hist]; val_l = [h[2] for h in hist]; val_m = [h[3] for h in hist]
    fig, ax = plt.subplots(1,2, figsize=(12,4))
    ax[0].plot(epn, tr_l, label='train'); ax[0].plot(epn, val_l, label='val'); ax[0].set_title('CE loss'); ax[0].legend(); ax[0].grid(True)
    ax[1].plot(epn, tr_m, label='train mIoU'); ax[1].plot(epn, val_m, label='val mIoU'); ax[1].set_title('mIoU'); ax[1].legend(); ax[1].grid(True)
    plt.suptitle('U-Net v2 training curves'); plt.tight_layout(); plt.savefig(os.path.join(RUNS, 'unet_v2_training_curves.png'), dpi=130); plt.show()
    print('Saved training curves →', os.path.join(RUNS, 'unet_v2_training_curves.png'))

## Cell 7 — Training curves

Plots and saves train/val CE loss and mIoU across all epochs.  
Saves → `runs/unet_v2/unet_v2_training_curves.png`.

In [ ]:
model.load_state_dict(torch.load(os.path.join(RUNS, 'best_unet_v2.pt'), map_location=DEVICE)); model.eval()
cmap = np.array([[0,0,0],[0,0,200],[0,200,0]], dtype=np.uint8)
n_show = min(6, len(val_pairs))
fig, axs = plt.subplots(n_show, 4, figsize=(16, 3*n_show))
if n_show==1: axs = [axs]
import random as R
R.seed(0); sel = R.sample(list(range(len(val_pairs))), n_show)
for r, j in enumerate(sel):
    ip, mp = val_pairs[j]; img0 = np.array(Image.open(ip).convert('RGB'))
    m0  = np.array(Image.open(mp).convert('L'))
    m0  = np.clip(m0, 0, 2).astype(int)
    H, W, _ = img0.shape
    t = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    p = t(Image.fromarray(img0)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pr = model(p).argmax(1).cpu().numpy()[0]
    g_small = np.array(Image.fromarray(m0.astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE), Image.Resampling.NEAREST))
    vis_gt = cmap[g_small]; vis_pr = cmap[pr]
    ovr = (np.array(img0) * 0.5 + np.array(Image.fromarray(vis_pr).resize((W,H)))*0.5).astype(np.uint8)
    for ax, im, ttl in zip(axs[r], [img0, vis_gt, vis_pr, ovr], ['ROI','GT 3c','Pred 3c','overlay']):
        ax.imshow(im if im.ndim==3 else im); ax.set_title(ttl, fontsize=9); ax.axis('off')
plt.suptitle('U-Net v2 validation sample'); plt.tight_layout(); plt.savefig(os.path.join(RUNS, 'unet_v2_predictions.png'), dpi=120); plt.show()
print('Saved prediction grid →', os.path.join(RUNS, 'unet_v2_predictions.png'))

## Cell 8 — Validation prediction grid

Reloads the **best** weights, runs inference on 6 random validation crops, and shows a 4-column grid: ROI image | ground-truth 3-class mask | predicted 3-class mask | blended overlay.  
Saves → `runs/unet_v2/unet_v2_predictions.png`.

In [ ]:
print('=== U-Net v2 summary ==='); print('Best val mIoU (mean of 3-class IoU):', best_m, 'at epoch', best_ep)
print('Weights:', os.path.join(RUNS, 'best_unet_v2.pt'))
print('Class names: 0=background 1=strawberry 2=peduncle'); print('Val confusion:\n', cm)


## Cell 9 — Final summary

Prints the best validation mIoU, the epoch it was achieved, weights path, class names, and the final confusion matrix. Use these numbers in the presentation and report.